# Coal SCADA Source

In [ ]:
from pathlib import Path
import traceback
import nemosis
import pandas as pd




Fetch and aggregate coal generation (SCADA) by region.

In [ ]:


def _process_month(start: str, end: str) -> pd.DataFrame:
    COAL_GROUPS = {
        "coal_mw_nsw": [
            "ER01", "ER02", "ER03", "ER04",
            "BW01", "BW02", "BW03", "BW04",
            "MP1", "MP2",
            "VP5", "VP6",
            "LD01", "LD02", "LD03", "LD04",
        ],
        "coal_mw_qld": [
            "STAN1", "STAN2", "STAN3", "STAN4",
            "GLAD1", "GLAD2", "GLAD3", "GLAD4", "GLAD5", "GLAD6",
            "TARONG1", "TARONG2", "TARONG3", "TARONG4",
            "TNPS1",
            "CALLB1", "CALLB2",
            "CALLC04", "CALLC2", "CALLC3",
            "MILLMERR1", "MILLMERR2",
            "KOGAN1",
        ],
        "coal_mw_vic": [
            "LYA1", "LYA2", "LYA3", "LYA4",
            "LOYYB1", "LOYYB2",
            "YW1", "YW2", "YW3", "YW4",
        ],
    }
    
    ALL_DUIDS   = [d for duids in COAL_GROUPS.values() for d in duids]
    DUID_TO_COL = {d: col for col, duids in COAL_GROUPS.items() for d in duids}

    chunk = nemosis.dynamic_data_compiler(
        start_time=start,
        end_time=end,
        table_name="DISPATCH_UNIT_SCADA",
        raw_data_location=str(Path("Pre_processing/nemosis_raw_dispatch_unit_scada")),
        select_columns=["SETTLEMENTDATE", "DUID", "SCADAVALUE"],
        filter_cols=["DUID"],
        filter_values=[ALL_DUIDS],
        fformat="feather",
        keep_csv=False,
    )
    chunk["SETTLEMENTDATE"] = pd.to_datetime(chunk["SETTLEMENTDATE"])
    chunk["SCADAVALUE"] = pd.to_numeric(chunk["SCADAVALUE"], errors="coerce").clip(lower=0)
    chunk["_col"] = chunk["DUID"].map(DUID_TO_COL)
    return chunk.groupby(["SETTLEMENTDATE", "_col"])["SCADAVALUE"].sum().unstack("_col")


def main():
    data_start = pd.Timestamp("2018/01/01")
    data_end   = pd.Timestamp("2026/01/01")

    processed_path = Path("Processed_data/3_dispatch_unit_scada.csv")
    already_processed = set()
    if processed_path.exists():
        existing_idx = pd.read_csv(processed_path, usecols=[0], parse_dates=True).iloc[:, 0]
        already_processed = set(pd.to_datetime(existing_idx).dt.to_period("M"))
        print(f"  Found {len(already_processed)} already-processed month(s) — will skip.", flush=True)

    total_months = pd.date_range(data_start, data_end - pd.Timedelta(days=1), freq="MS")
    monthly_dfs = []

    for i, month in enumerate(total_months):
        if month.to_period("M") in already_processed:
            print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} skipping (already processed).", flush=True)
            continue
        start = month.strftime("%Y/%m/%d %H:%M:%S")
        end   = (month + pd.offsets.MonthEnd(0) + pd.Timedelta(days=1)).strftime("%Y/%m/%d %H:%M:%S")
        print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} fetching...", flush=True)
        try:
            monthly_dfs.append(_process_month(start, end))
            for f in Path("Pre_processing/nemosis_raw_dispatch_unit_scada").glob(f"*{month.strftime('%Y%m')}*.feather"):
                f.unlink()
            print(f"  {month:%Y-%m} raw files deleted.", flush=True)
        except Exception:
            print(f"  WARNING: {month:%Y-%m} failed:\n{traceback.format_exc()}", flush=True)

    if not monthly_dfs:
        print("No new months to process.", flush=True)
        return pd.read_csv(processed_path, index_col=0, parse_dates=True).tail()

    df = pd.concat(monthly_dfs).sort_index()
    df = df.resample("5min").mean()
    df = df[~df.index.duplicated(keep="last")]

    if processed_path.exists():
        existing = pd.read_csv(processed_path, index_col=0, parse_dates=True)
        existing.index = pd.to_datetime(existing.index, format="mixed")
        df = pd.concat([existing, df])
        df = df[~df.index.duplicated(keep="last")].sort_index()

    df.index.name = "SETTLEMENTDATE"
    df.to_csv(processed_path)
    return df.tail()

main()


  Found 97 already-processed month(s) — will skip.
  1/96 2018-01 skipping (already processed).
  2/96 2018-02 skipping (already processed).
  3/96 2018-03 skipping (already processed).
  4/96 2018-04 skipping (already processed).
  5/96 2018-05 skipping (already processed).
  6/96 2018-06 skipping (already processed).
  7/96 2018-07 skipping (already processed).
  8/96 2018-08 skipping (already processed).
  9/96 2018-09 skipping (already processed).
 10/96 2018-10 skipping (already processed).
 11/96 2018-11 skipping (already processed).
 12/96 2018-12 skipping (already processed).
 13/96 2019-01 skipping (already processed).
 14/96 2019-02 skipping (already processed).
 15/96 2019-03 skipping (already processed).
 16/96 2019-04 skipping (already processed).
 17/96 2019-05 skipping (already processed).
 18/96 2019-06 skipping (already processed).
 19/96 2019-07 skipping (already processed).
 20/96 2019-08 skipping (already processed).
 21/96 2019-09 skipping (already processed).
 22/

,coal_mw_nsw,coal_mw_qld,coal_mw_vic
SETTLEMENTDATE,,,
2025-12-31 23:40:00,4325.45508,351.2500,1998.500
2025-12-31 23:45:00,4310.48899,350.1250,2020.000
2025-12-31 23:50:00,4350.67520,351.2500,2028.750
2025-12-31 23:55:00,4313.68736,351.2500,2010.500
2026-01-01 00:00:00,4323.42876,349.9375,1979.375
